# Controls: what our probe reads, and which writes move which model

Companion to [`probe_transfer.ipynb`](probe_transfer.ipynb), which established that **our probe and
our editor, unmodified, reproduce Li et al.'s intervention on their model** (null 2.723 → 0.016
against their published 2.68 → 0.12). That cleared the *implementation*. This notebook asks the two
questions that clearing sat on top of:

1. **Is our probe reading a learned state, or is it reading the observation?** If the latter, every
   editability claim that leans on "high probe R² means a meaningful representation" is weaker than
   it looks.
2. **Which write mechanisms actually move their model** — and does the ranking match what we see on
   ours?

Everything here is one place to re-derive the numbers quoted in
`../../../../research/scratch/2026-08-21-probe-reality-checks.md` and
`2026-08-21-linear-direction-interventions.md`.

**Two models, and it matters which is which in every table below.**

| | code | what |
|---|---|---|
| **ours** | `runs/transformers/W16` | this repo's transformer world model — `d_model` 256, 4 layers, 4 heads, RoPE, band-causal window 16, `Linear(128,256)+ReLU` encoder. Dataset `4_fixed_refl_inview` |
| **theirs** | `runs/othello_transfer/gpt_synthetic.ckpt` | Li et al.'s Othello-GPT — 8 blocks, 8 heads, `d_model` 512, full causal. Provenance and the three-way verification in [`OTHELLO_TRANSFER_RUNS.md`](OTHELLO_TRANSFER_RUNS.md) |

Sections 1–3 are **our** model. Section 4 is **theirs**.

**What prompted this.** Sevan's hypothesis that discworld's observation manifold may be
low-dimensional enough that a probe reads the *observation* rather than a learned state — in which
case the editability negative would be far less interesting than it looks. Section 2 puts a number
on it and section 3 shows how much of what remains is the architecture rather than training.


## Definitions

**Probe.** `othello_gpt/othello_probe.fit_probe` throughout, unchanged — linear is exact `lstsq`,
MLP is one hidden layer of 512 trained 200 epochs. Held out **by sequence**, never by frame
(`harness/ANALYSIS.md` §2). Target is object **position**, 4 dims (2 objects × x, y). Reported as
held-out **R² against the train mean**, ↑.

**Residual point ℓ** — the stream after ℓ blocks; ℓ = 0 is the encoder port. `W16` has 4 layers, so
**5 points**; Othello-GPT has 8 layers, so **9**.

| section | measures | why it is the right control |
|---|---|---|
| **1 · probe-data scaling** | R² vs number of sequences the probe is fit on | our editability numbers fit probes on **1500 sequences (48k rows)**; Li et al. fit theirs on ~140k games (**≈6.7M rows**). A 140× gap is a live alternative explanation for the negative |
| **2 · observation baseline** | R² from the **raw observation** the model receives, by window length | if the observation already determines position, a high latent R² is not evidence of a learned state. Window length separates "the frame shows it" from "integrating frames shows it" |
| **3 · random-weight baseline** | R² from an **untrained** network, identical architecture and data | Li et al.'s own `--random` arm. Random weights give a smooth deterministic function, not a random one — probes read a surprising amount from random features |

**Section 4 — interventions on *their* model.** All write to the residual stream at the last
sequence position and let the network recompute.

| mechanism | write | source |
|---|---|---|
| **Nanda direction addition** | `x ← x + α·p_d`, `p_d` = the linear probe's weight column for the target direction | Nanda, Lee & Wattenberg 2309.00941 §4.1 — one vector addition, no gradients |
| **ours: pseudoinverse injection** | `Δ = A⁺(target − (Az+b))`, minimum-norm, null-space preserving | `pim.editors.probe_steering.inject_state`, **unmodified** |

| metric | definition | units | better |
|---|---|---|---|
| **Li error** | top-*N* predicted moves vs the post-flip legal set, false pos + false neg, *N* = number of legal moves | errors | ↓ |
| **Li error vs pre-flip** | the same against the **pre**-flip legal set — the guard. A null intervention is low here and high on the metric above; a successful edit is the reverse; an arm that **destroyed** the model is high on **both** | errors | ↑ for a real edit |
| **Edit Index (union)** | this repo's `(d_uned − d_edit)/(d_uned + d_edit)`, `d_·` = RMSE of the predicted move distribution against uniform-over-legal, over the union of the two legal sets | −1…+1 | ↑ |
| **‖Δx‖/‖x‖** | write size relative to the activation — the quantity that makes different α parameterisations comparable | ratio | — |

**Published values to hit:** Li et al. null **2.68** → **0.12**; Nanda linear addition **0.10**;
Li nonlinear probe **1.7%** error, linear **20.4%**.


In [ ]:
# [1] Setup. Recurring computations live in `controls_lib.py` (our model) and
#     `linear_intervention.py` (their model), both beside this notebook.
#     `othello_gpt/pipeline.load` resolves `runs/` and `datasets/` relative to the CWD, so the
#     kernel is moved to the repo root; every path this notebook writes is absolute.
import json, os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

THREAD = Path.cwd().resolve()
REPO = THREAD.parents[3]
for p in (str(THREAD), str(REPO), str(REPO / "scripts"),
          str(THREAD.parent / "othello_gpt")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO)
import controls_lib as ctl
import pipeline as pl
from pim.figures.theme import PALETTE, style_ax

FIGDIR = REPO / "runs" / "othello_transfer" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

SCALES = (1500, 5000, 15000, 45000, 90000)   # sequences the probe is fit on
WINDOWS = (1, 2, 4, 8, 16)
N_CTRL = 5000
SEED = 0

# published / previously established numbers, cited not recomputed
REF = {"li_nonlinear_probe": 1.7, "li_linear_probe": 20.4, "li_null": 2.68, "li_best": 0.12,
       "nanda_linear_add": 0.10, "w16_published_mlp_r2": 0.9349,
       "obs_clean_1frame_linear": 0.259, "obs_clean_1frame_mlp": 0.754}

torch.manual_seed(SEED)
np.random.seed(SEED)
RESULTS = {}
ours = pl.load("W16").model
print(f"device {pl.DEVICE}   ·  thread {THREAD.name}   ·  cwd {Path.cwd().name}")
print(f"OUR model  W16: d_model {ours.cfg.d_model}, {ours.cfg.n_layers} layers, "
      f"{ours.cfg.n_heads} heads, window {ours.cfg.window} -> "
      f"{ours.cfg.n_layers + 1} residual points")
print(f"probe: hidden {ctl.HIDDEN}, {ctl.EPOCHS} epochs, held out BY SEQUENCE, "
      f"position target at residual point {ctl.POINT}")

In [ ]:
# [2] SECTION 1 — OUR model. Does probe quality improve at Li et al.'s probe-data scale?
#     Anchor first: the thread's published 0.9349 was fit on 1500 TEST sequences. Reproducing it
#     confirms the curve below is measuring the same thing the thread quotes. ~3 min.
t0 = time.time()
anchor = ctl.probe_scaling(ours, [1500], split="test", log=lambda s: None)[0]
print(f"anchor (test split, 1500 seq): linear {anchor['linear']:.4f}  MLP {anchor['mlp']:.4f}"
      f"   — thread's published MLP R² is {REF['w16_published_mlp_r2']}")
assert abs(anchor["mlp"] - REF["w16_published_mlp_r2"]) < 0.02, "does not reproduce the published probe"

scaling = ctl.probe_scaling(ours, SCALES, split="train")
print(f"\n{time.time() - t0:.0f}s")
RESULTS["probe_scaling"] = scaling
RESULTS["probe_anchor"] = anchor

rows = ["| sequences | train rows | linear R² | MLP R² |", "|---|---|---|---|",
        f"| *1,500 (test split — the published anchor)* | *{anchor['rows']:,}* | "
        f"*{anchor['linear']:.4f}* | *{anchor['mlp']:.4f}* |"]
for r in scaling:
    rows.append(f"| {r['n_seq']:,} | {r['rows']:,} | {r['linear']:.4f} | **{r['mlp']:.4f}** |")
display(Markdown("**Table 1 — probe quality vs probe training data (our model).**\n\n" + "\n".join(rows)))

g_mlp = scaling[-1]["mlp"] - scaling[0]["mlp"]
steps = [scaling[i + 1]["mlp"] - scaling[i]["mlp"] for i in range(len(scaling) - 1)]
print(f"\n{scaling[-1]['rows'] / scaling[0]['rows']:.0f}x more probe data buys "
      f"{g_mlp:+.4f} MLP R² ({scaling[0]['mlp']:.4f} -> {scaling[-1]['mlp']:.4f})")
print("successive steps: " + ", ".join(f"{s:+.4f}" for s in steps))
print(f"linear moves {scaling[-1]['linear'] - scaling[0]['linear']:+.4f}")
print(f"\nLi et al. fit their probes on ~6.7M rows; we saturate well short of that.")


In [ ]:
# [3] SECTION 2 — OUR model. How much position is in the RAW observation, and how much does
#     integrating frames add? This is the control for the low-dimensional-manifold hypothesis.
win = ctl.obs_window(WINDOWS, n=N_CTRL)
RESULTS["obs_window"] = win

rows = ["| window (frames) | input dim | linear R² | MLP R² |", "|---|---|---|---|"]
for r in win:
    rows.append(f"| {r['window']} | {r['dim']} | {r['linear']:.4f} | {r['mlp']:.4f} |")
display(Markdown("**Table 2 — position from the raw (noisy) observation the model receives.**\n\n"
                 + "\n".join(rows)))
print(f"2026-08-05 reference (clean_obs, ONE frame): linear {REF['obs_clean_1frame_linear']} / "
      f"MLP {REF['obs_clean_1frame_mlp']}")
print(f"\ntemporal integration, 1 -> {WINDOWS[-1]} frames: "
      f"linear {win[-1]['linear'] - win[0]['linear']:+.4f}, MLP {win[-1]['mlp'] - win[0]['mlp']:+.4f}")
print("⚠ Long windows are under-determined here — the input dimension outruns the row count, and")
print("   the MLP starts to fall. Read the trend from the short windows, not the longest one.")


In [ ]:
# [4] SECTION 3 — OUR model. Li et al.'s own `--random` control: identical architecture and data,
#     random weights, two seeds, every residual point. ~2 min.
t0 = time.time()
ri = ctl.random_init(ours, n=N_CTRL, seeds=(0, 1))
print(f"\n{time.time() - t0:.0f}s")
RESULTS["random_init"] = ri

pts = [r["point"] for r in ri["trained"]]
rows = ["| residual point | " + " | ".join(str(p) for p in pts) + " |", "|---|" + "---|" * len(pts)]
for lab, key, fam in (("trained, linear", "trained", "linear"), ("random, linear", "random", "linear"),
                      ("trained, MLP", "trained", "mlp"), ("random, MLP", "random", "mlp")):
    rows.append(f"| {lab} | " + " | ".join(f"{r[fam]:.3f}" for r in ri[key]) + " |")
display(Markdown("**Table 3 — trained vs randomly initialised, by residual point (our model).**\n\n"
                 + "\n".join(rows)))

bt_l = max(r["linear"] for r in ri["trained"]); br_l = max(r["linear"] for r in ri["random"])
bt_m = max(r["mlp"] for r in ri["trained"]); br_m = max(r["mlp"] for r in ri["random"])
obs_l, obs_m = win[-1]["linear"], win[-1]["mlp"]
print(f"\n{'':<34}{'linear':>10}{'MLP':>10}")
for lab, a, b in (("raw observation (16-frame window)", obs_l, obs_m),
                  ("random-init latent, best point", br_l, br_m),
                  ("trained latent, best point", bt_l, bt_m)):
    print(f"{lab:<34}{a:>10.3f}{b:>10.3f}")
print(f"\narchitecture contributes  linear {br_l - obs_l:+.3f}   MLP {br_m - obs_m:+.3f}")
print(f"training contributes      linear {bt_l - br_l:+.3f}   MLP {bt_m - br_m:+.3f}")
print("\nThe DEPTH TREND separates them where the levels do not: random decodability declines with")
print("depth while trained decodability rises. Random features degrade as they are mixed; learned")
print("features accumulate. No random projection reproduces that.")
RESULTS["contributions"] = {"obs_linear": obs_l, "obs_mlp": obs_m, "rand_linear": br_l,
                            "rand_mlp": br_m, "trained_linear": bt_l, "trained_mlp": bt_m}


In [ ]:
# [5] Fig 1 — sections 1–3 together. Every panel is OUR model, absolute R², with the baseline it
#     should be read against drawn on the same axis.
fig, axes = plt.subplots(1, 3, figsize=(17.0, 4.8), facecolor="white")

r_ = [s["rows"] for s in scaling]
axes[0].plot(r_, [s["mlp"] for s in scaling], color=PALETTE[0], ls="-", marker="o", lw=2,
             label="MLP (512 hidden)")
axes[0].plot(r_, [s["linear"] for s in scaling], color=PALETTE[1], ls="--", marker="s", lw=2,
             label="linear")
axes[0].axhline(REF["w16_published_mlp_r2"], color="0.4", ls=":", lw=1.6,
                label=f"thread's published MLP R² ({REF['w16_published_mlp_r2']})")
axes[0].axvline(6.7e6, color="0.7", ls="-.", lw=1.6, label="Li et al.'s probe-data scale (~6.7M rows)")
axes[0].set_xscale("log")
axes[0].set_xlabel("rows the probe is fit on")
axes[0].set_title("(a) probe data — saturates far short of theirs")

w_ = [r["window"] for r in win]
axes[1].plot(w_, [r["mlp"] for r in win], color=PALETTE[0], ls="-", marker="o", lw=2, label="MLP")
axes[1].plot(w_, [r["linear"] for r in win], color=PALETTE[1], ls="--", marker="s", lw=2, label="linear")
axes[1].axhline(bt_m, color=PALETTE[0], ls=":", lw=1.6, label=f"trained latent, MLP ({bt_m:.3f})")
axes[1].axhline(bt_l, color=PALETTE[1], ls=":", lw=1.6, label=f"trained latent, linear ({bt_l:.3f})")
axes[1].set_xscale("log", base=2)
axes[1].set_xticks(w_); axes[1].set_xticklabels([str(w) for w in w_]); axes[1].minorticks_off()
axes[1].set_xlabel("observation window (frames)")
axes[1].set_title("(b) raw observation vs the trained latent")

axes[2].plot(pts, [r["mlp"] for r in ri["trained"]], color=PALETTE[0], ls="-", marker="o", lw=2,
             label="trained, MLP")
axes[2].plot(pts, [r["mlp"] for r in ri["random"]], color=PALETTE[0], ls="--", marker="^", lw=1.6,
             alpha=0.85, label="random init, MLP")
axes[2].plot(pts, [r["linear"] for r in ri["trained"]], color=PALETTE[1], ls="-", marker="o", lw=2,
             label="trained, linear")
axes[2].plot(pts, [r["linear"] for r in ri["random"]], color=PALETTE[1], ls="--", marker="^", lw=1.6,
             alpha=0.85, label="random init, linear")
axes[2].axhline(obs_m, color="0.35", ls=":", lw=1.6, label=f"raw observation, MLP ({obs_m:.3f})")
axes[2].axhline(obs_l, color="0.65", ls=":", lw=1.6, label=f"raw observation, linear ({obs_l:.3f})")
axes[2].set_xticks(pts)
axes[2].set_xlabel("residual point (0 = encoder port)")
axes[2].set_title("(c) trained vs random — the depth trend is the signal")

for ax in axes:
    ax.set_ylabel("held-out position R²")
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=7.5, handlelength=2.6, loc="lower right")
    style_ax(ax)
fig.suptitle("Fig 1 — what our probe is reading (model: W16, dataset 4_fixed_refl_inview)", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94), w_pad=2.0)
fig.savefig(FIGDIR / "fig_controls_probe.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# [6] SECTION 4 — THEIR model. Nanda's linear-direction addition vs our pseudoinverse injection,
#     on Li et al.'s Othello-GPT, their 1001-case benchmark, our linear mine/theirs probe.
#     `linear_intervention.run` is unchanged and its `pinv` branch calls
#     `pim.editors.probe_steering.inject_state` directly. ~5 min.
import linear_intervention as li
import othello_data as od
import transfer_pipeline as tp

t0 = time.time()
shim = tp.load_model()
bench = od.load_benchmark()
lin_probes = li.load_linear_probes()
cur_lab, tgt_lab = li.case_targets(bench)
uns = od.scorecard(tp.unsteered(shim, bench), bench)
ALLP = set(range(tp.N_POINTS))
print(f"\nnull intervention: Li error {uns['li_error_vs_post']:.3f} "
      f"(Nanda Table 2 reports 2.723)   Edit Index {uns['edit_index_union']:+.3f}")
assert abs(uns["li_error_vs_post"] - 2.723) < 0.02, "not the same benchmark/metric as theirs"

A_ADD = (0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35, 0.5)
A_PINV = (0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0)
SW = {}
for mode, alphas, tag in (("add", A_ADD, "Nanda: add target direction"),
                          ("add", A_ADD, "Nanda: add (target - current)"),
                          ("pinv", A_PINV, "OURS: inject_state, all 9 points")):
    sub = tag.endswith("current)")
    for a in alphas:
        _, c = li.run(shim, bench, lin_probes, mode, a, ALLP, tgt_lab, cur_lab, sub)
        SW[(tag, a)] = c
best = {t: min([k for k in SW if k[0] == t], key=lambda k: SW[k]["li_error_vs_post"])
        for t in {k[0] for k in SW}}
print(f"\n{'arm':<34} {'best a':>7} {'|dx|/|x|':>9} {'Li post':>8} {'Li pre':>8} {'EI union':>9} {'legal':>6}")
print(f"{'no intervention':<34} {'—':>7} {'—':>9} {uns['li_error_vs_post']:>8.3f} "
      f"{uns['li_error_vs_pre']:>8.3f} {uns['edit_index_union']:>+9.3f} {uns['legal_mass']:>6.3f}")
for t, k in best.items():
    c = SW[k]
    print(f"{t:<34} {k[1]:>7} {c['write_ratio']:>9.3f} {c['li_error_vs_post']:>8.3f} "
          f"{c['li_error_vs_pre']:>8.3f} {c['edit_index_union']:>+9.3f} {c['legal_mass']:>6.3f}")
print(f"\n{time.time() - t0:.0f}s   ·  published: Li et al. {REF['li_best']}, "
      f"Nanda linear addition {REF['nanda_linear_add']}")
RESULTS["othello_sweep"] = {f"{t}|{a}": {k: v for k, v in c.items() if not k.endswith("per_case")}
                            for (t, a), c in SW.items()}
RESULTS["othello_unsteered"] = {k: v for k, v in uns.items() if not k.endswith("per_case")}


In [ ]:
# [7] SECTION 4b — THEIR model. The same two writes applied at ONE residual point only.
#     This is the decisive comparison: our pseudoinverse re-reads the probe at every layer and so
#     re-imposes "hold all 64 tiles" against the already-edited stream, undoing its own edit.
#     Nanda's direction does not depend on x and cannot do that. ~2 min.
t0 = time.time()
SL = {}
for mode, alphas, tag in (("pinv", (0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0), "OURS pseudoinverse"),
                          ("add", (0.05, 0.12, 0.18, 0.25, 0.35, 0.5, 0.75, 1.0), "Nanda addition")):
    for ell in range(tp.N_POINTS):
        bc = None
        for a in alphas:
            _, c = li.run(shim, bench, lin_probes, mode, a, {ell}, tgt_lab, cur_lab)
            if bc is None or c["li_error_vs_post"] < bc["li_error_vs_post"]:
                bc, ba = c, a
        SL[(tag, ell)] = {"alpha": ba, **{k: v for k, v in bc.items() if not k.endswith("per_case")}}
print(f"{time.time() - t0:.0f}s\n")

hdr = "| point written | " + " | ".join(str(p) for p in range(tp.N_POINTS)) + " | **all 9** |"
rows = [hdr, "|---|" + "---|" * (tp.N_POINTS + 1)]
for tag, allkey in (("OURS pseudoinverse", "OURS: inject_state, all 9 points"),
                    ("Nanda addition", "Nanda: add target direction")):
    rows.append(f"| {tag} — Li error ↓ | "
                + " | ".join(f"{SL[(tag, p)]['li_error_vs_post']:.3f}" for p in range(tp.N_POINTS))
                + f" | **{SW[best[allkey]]['li_error_vs_post']:.3f}** |")
    rows.append(f"| {tag} — Edit Index | "
                + " | ".join(f"{SL[(tag, p)]['edit_index_union']:+.3f}" for p in range(tp.N_POINTS))
                + f" | **{SW[best[allkey]]['edit_index_union']:+.3f}** |")
display(Markdown("**Table 4 — writing at ONE residual point vs all nine (their model).** "
                 "Best α per cell.\n\n" + "\n".join(rows)))

bp = min(range(tp.N_POINTS), key=lambda p: SL[("OURS pseudoinverse", p)]["li_error_vs_post"])
c = SL[("OURS pseudoinverse", bp)]
print(f"OUR pseudoinverse, best single point = {bp} (α {c['alpha']}): Li error "
      f"{c['li_error_vs_post']:.3f}, Edit Index {c['edit_index_union']:+.3f}, "
      f"legal mass {c['legal_mass']:.3f}")
print(f"  vs the SAME editor written at all 9 points: "
      f"{SW[best['OURS: inject_state, all 9 points']]['li_error_vs_post']:.3f}")
print("\nMulti-layer HELPS Nanda's fixed direction and HURTS our recomputed injection. That")
print("asymmetry is the evidence that the 'hold everything else' constraint, re-imposed against")
print("the already-edited stream, is what cancels the edit.")
RESULTS["single_layer"] = {f"{t}|{p}": v for (t, p), v in SL.items()}


In [ ]:
# [8] Fig 2 — section 4. Absolute Li error, log scale, with the null and both published values
#     on the same axis; and the depth profile that carries the mechanism.
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), facecolor="white")
STY = {"Nanda: add target direction": dict(color=PALETTE[0], ls="-", marker="o"),
       "Nanda: add (target - current)": dict(color=PALETTE[3], ls="-", marker="^"),
       "OURS: inject_state, all 9 points": dict(color=PALETTE[1], ls="--", marker="s")}
for tag, st in STY.items():
    ks = sorted([k for k in SW if k[0] == tag], key=lambda k: k[1])
    axes[0].plot([SW[k]["write_ratio"] for k in ks], [SW[k]["li_error_vs_post"] for k in ks],
                 label=tag, lw=2, ms=6, **st)
axes[0].axhline(uns["li_error_vs_post"], color="0.35", ls=":", lw=1.8,
                label=f"no intervention ({uns['li_error_vs_post']:.2f})")
axes[0].axhline(REF["li_best"], color="0.6", ls="--", lw=1.6, label=f"Li et al. ({REF['li_best']})")
axes[0].axhline(REF["nanda_linear_add"], color="0.8", ls="-.", lw=1.6,
                label=f"Nanda linear addition ({REF['nanda_linear_add']})")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("write size ‖Δx‖/‖x‖  — the comparable axis across parameterisations")
axes[0].set_ylabel("Li error vs the post-flip legal set (log)  — lower is better")
axes[0].set_title("(a) written at every residual point")

pp = list(range(tp.N_POINTS))
for tag, st in (("OURS pseudoinverse", dict(color=PALETTE[1], ls="--", marker="s")),
                ("Nanda addition", dict(color=PALETTE[0], ls="-", marker="o"))):
    axes[1].plot(pp, [SL[(tag, p)]["li_error_vs_post"] for p in pp], label=f"{tag}, single point",
                 lw=2, ms=6, **st)
    allkey = ("OURS: inject_state, all 9 points" if tag.startswith("OURS")
              else "Nanda: add target direction")
    axes[1].axhline(SW[best[allkey]]["li_error_vs_post"], color=st["color"], ls=":", lw=1.6,
                    alpha=0.8, label=f"{tag}, ALL 9 points")
axes[1].axhline(uns["li_error_vs_post"], color="0.35", ls=":", lw=1.8, label="no intervention")
axes[1].axhline(REF["li_best"], color="0.6", ls="--", lw=1.6, label=f"Li et al. ({REF['li_best']})")
axes[1].set_yscale("log")
axes[1].set_xticks(pp)
axes[1].set_xlabel("the single residual point written to")
axes[1].set_ylabel("Li error vs the post-flip legal set (log)")
axes[1].set_title("(b) one point at a time — multi-layer helps one method and ruins the other")

for ax in axes:
    ax.legend(fontsize=7.5, handlelength=2.6)
    style_ax(ax)
fig.suptitle("Fig 2 — probe-derived writes on Li et al.'s Othello-GPT (their 1001-case benchmark)",
             fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94), w_pad=2.0)
fig.savefig(FIGDIR / "fig_controls_intervention.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# [9] Summary — rendered from the values computed above rather than transcribed, so it cannot go
#     stale against the tables. Each claim is stated with the control that licenses it.
pi = SL[("OURS pseudoinverse", bp)]
na = SW[best["Nanda: add target direction"]]
oa = SW[best["OURS: inject_state, all 9 points"]]
lines = [
    "## What the four controls establish",
    "",
    "### Sections 1–3 · OUR model (`W16`), position probe",
    "",
    f"**1. Probe training data is not the limiting factor.** "
    f"{scaling[-1]['rows'] / scaling[0]['rows']:.0f}x more probe data moves the MLP {g_mlp:+.4f} R² "
    f"({scaling[0]['mlp']:.4f} -> {scaling[-1]['mlp']:.4f}); the last step buys {steps[-1]:+.4f}. "
    f"Li et al. fit theirs on ~140x our rows and we are already flat well short of that. The "
    f"editability negative is not a starved probe.",
    "",
    f"**2. Most of the decodability is in the observation, not the latent.** The raw "
    f"{WINDOWS[-1]}-frame observation gives linear {obs_l:.3f} / MLP {obs_m:.3f}; the best trained "
    f"residual point gives {bt_l:.3f} / {bt_m:.3f}. Training adds {bt_l - br_l:+.3f} linear / "
    f"{bt_m - br_m:+.3f} MLP **over an untrained network of the same architecture**, which itself "
    f"adds {br_l - obs_l:+.3f} / {br_m - obs_m:+.3f} over the observation. A high probe R² is "
    f"therefore weak evidence on its own for a learned state — Sevan's low-dimensional-manifold "
    f"hypothesis survives this test.",
    "",
    "**3. The depth trend is not explained by the architecture.** Random-init decodability *falls* "
    "with depth while trained decodability *rises*: random features degrade as they are mixed, "
    "learned features accumulate. Levels alone cannot separate the two hypotheses; the sign of the "
    "slope can, and it favours something being built across layers.",
    "",
    "### Section 4 · THEIR model (Li et al.'s Othello-GPT), their 1001-case benchmark",
    "",
    f"**4. Our editor is not the problem — on their model it is the strongest write tested.** "
    f"Written at the single best residual point (point {bp}, α = {pi['alpha']}), our *unmodified* "
    f"`inject_state` reaches Li error **{pi['li_error_vs_post']:.3f}** / Edit Index "
    f"**{pi['edit_index_union']:+.3f}**, against the null's {uns['li_error_vs_post']:.3f} / "
    f"{uns['edit_index_union']:+.3f}, Nanda's addition at the same point "
    f"({SL[('Nanda addition', bp)]['li_error_vs_post']:.3f}), and the published {REF['li_best']} "
    f"(Li et al.) / {REF['nanda_linear_add']} (Nanda et al.).",
    "",
    f"**5. Depth of application flips the ranking, and that is the mechanism.** The *same* "
    f"pseudoinverse written at all nine points degrades to {oa['li_error_vs_post']:.3f} from "
    f"{pi['li_error_vs_post']:.3f} at one, while Nanda's fixed direction *improves* with more points "
    f"({SL[('Nanda addition', bp)]['li_error_vs_post']:.3f} -> {na['li_error_vs_post']:.3f}). A "
    f"recomputed injection re-reads the probe against the already-edited stream and re-imposes "
    f"\"hold the other 63 tiles\", cancelling its own edit; a fixed direction does not depend on x "
    f"and cannot do that. Multi-layer application is a property of the **editor parameterisation**, "
    f"not of the model.",
    "",
    "### Consequence for the thread",
    "",
    "The two confounds this notebook was built to remove are removed: neither the probe "
    "implementation, nor the probe's training data, nor our editor explains discworld's editability "
    "failure — the same code succeeds on a different architecture in a different environment "
    "(established in `probe_transfer.ipynb`, sharpened here). What section 2 *does* put in play is "
    "the **target**: our probe may be reading the observation rather than a state, which is grounds "
    "to doubt the premise of the measurement rather than the editor. "
    "`research/directions/othello-architecture-on-discworld.md` is the run that separates the two "
    "remaining confounds — architecture and training-data scale.",
]
display(Markdown("\n".join(lines)))

In [ ]:
# [10] Serialise. Written to its own file — `results.json` beside it belongs to
#      `probe_transfer.ipynb` and must not be clobbered. Per-case arrays are kept: filtering
#      lists out on the way to JSON has silently dropped curves a plot depended on before
#      (`harness/ANALYSIS.md` §1), so the sweep is written whole.
out = REPO / "runs" / "othello_transfer" / "results_controls.json"
RESULTS["config"] = {
    "seed": SEED, "scales": list(SCALES), "windows": list(WINDOWS), "n_ctrl": N_CTRL,
    "probe_hidden": ctl.HIDDEN, "probe_epochs": ctl.EPOCHS, "probe_point": ctl.POINT,
    "our_model": "runs/transformers/W16", "our_dataset": "4_fixed_refl_inview",
    "their_model": "runs/othello_transfer/gpt_synthetic.ckpt",
    "reference_values": REF,
}


def _plain(o):
    if isinstance(o, dict):
        return {str(k): _plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_plain(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


out.write_text(json.dumps(_plain(RESULTS), indent=1))
print(f"wrote {out.relative_to(REPO)}  ({out.stat().st_size:,} bytes)")
print("figures: " + ", ".join(sorted(p.name for p in FIGDIR.glob("fig_controls_*.png"))))